# PyTorch → Keras : entraîner, sauvegarder, recharger

Ce notebook montre le cycle complet entre les deux implémentations hexagonales :

1. entraîner un petit modèle avec **`pytorch-hexagdly`** ;
2. sauvegarder ses poids, en `.pth` puis en `.npz` ;
3. reconstruire l'équivalent **`keras-hexagdly`** et y charger les poids ;
4. vérifier que les deux modèles donnent les mêmes sorties.

Le point important de l'étape 3 : le chargeur ne connaît que la **disposition**
des poids (PyTorch met les canaux en premier, Keras en dernier). Il ne devine
pas quelle couche PyTorch correspond à quelle couche Keras — les deux graphes
sont construits séparément, donc la correspondance est déclarée explicitement.

In [1]:
import sys, pathlib
# les deux dépôts, côte à côte sur le bureau
sys.path.insert(0, str(pathlib.Path.home() / "Desktop/pytorch-hexagdly/src"))
sys.path.insert(0, str(pathlib.Path.home() / "Desktop/keras-hexagdly/src"))

import numpy as np
import torch
import torch.nn as nn
import pytorch_hexagdly as phex

torch.manual_seed(0)
np.random.seed(0)
print("torch", torch.__version__)

torch 2.12.0+cu130


## 1. Entraîner un petit modèle

Tâche jouet : distinguer une tache hexagonale claire d'un fond bruité, sur une
grille 16×16. Assez pour que les poids soient non triviaux, assez petit pour
tourner en quelques secondes.

On utilise `share_neighbors="ring"` — le partage de poids par anneau, propre à
ce dépôt (le `hexagdly` amont ne l'a pas). Le chargeur gère les deux cas.

In [2]:
H = W = 16
N_TRAIN, N_TEST = 512, 128

def make_dataset(n, rng):
    """n images (n, 1, H, W) : moitié avec une tache centrale, moitié sans."""
    x = rng.standard_normal((n, 1, H, W)).astype(np.float32) * 0.4
    y = (rng.random(n) < 0.5).astype(np.float32)
    yy, xx = np.mgrid[0:H, 0:W]
    for i in np.flatnonzero(y):
        cy, cx = rng.integers(4, H - 4), rng.integers(4, W - 4)
        blob = np.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / 6.0)
        x[i, 0] += 2.0 * blob.astype(np.float32)
    return torch.from_numpy(x), torch.from_numpy(y[:, None])

rng = np.random.default_rng(0)
x_train, y_train = make_dataset(N_TRAIN, rng)
x_test,  y_test  = make_dataset(N_TEST,  rng)
print("train", tuple(x_train.shape), "test", tuple(x_test.shape))

train (512, 1, 16, 16) test (128, 1, 16, 16)


In [3]:
class HexNet(nn.Module):
    def __init__(self, share="ring"):
        super().__init__()
        self.conv1 = phex.Conv2d(1, 4, kernel_size=1, stride=1, bias=True, share_neighbors=share)
        self.act1  = nn.ReLU()
        self.pool  = phex.MaxPool2d(kernel_size=1, stride=2)
        self.conv2 = phex.Conv2d(4, 8, kernel_size=1, stride=1, bias=True, share_neighbors=share)
        self.act2  = nn.ReLU()
        self.head  = nn.Linear(8, 1)

    def forward(self, x):
        x = self.act1(self.conv1(x))
        x = self.pool(x)
        x = self.act2(self.conv2(x))
        x = x.mean(dim=(2, 3))          # global average pooling
        return self.head(x)

net = HexNet()
print(net)
print("paramètres :", sum(p.numel() for p in net.parameters()))

HexNet(
  (conv1): Conv2d(1, 4, kernel_size=1, stride=1)
  (act1): ReLU()
  (pool): MaxPool2d(kernel_size=1, stride=2)
  (conv2): Conv2d(4, 8, kernel_size=1, stride=1)
  (act2): ReLU()
  (head): Linear(in_features=8, out_features=1, bias=True)
)
paramètres : 93


In [4]:
opt = torch.optim.Adam(net.parameters(), lr=5e-3)
lossf = nn.BCEWithLogitsLoss()

for epoch in range(15):
    net.train()
    perm = torch.randperm(N_TRAIN)
    tot = 0.0
    for i in range(0, N_TRAIN, 64):
        idx = perm[i:i + 64]
        opt.zero_grad()
        loss = lossf(net(x_train[idx]), y_train[idx])
        loss.backward()
        opt.step()
        tot += loss.item() * len(idx)
    if epoch % 3 == 0 or epoch == 14:
        net.eval()
        with torch.no_grad():
            acc = ((net(x_test) > 0).float() == y_test).float().mean().item()
        print(f"epoch {epoch:2d}  loss {tot / N_TRAIN:.4f}  test acc {acc:.3f}")

epoch  0  loss 1.3346  test acc 0.227


epoch  3  loss 0.4872  test acc 0.938


epoch  6  loss 0.2716  test acc 1.000


epoch  9  loss 0.1654  test acc 1.000


epoch 12  loss 0.1036  test acc 1.000


epoch 14  loss 0.0786  test acc 1.000


## 2. Sauvegarder

Deux formats, pour deux usages.

Le `.pth` est le format naturel de PyTorch. Le `.npz` est utile si la machine
qui fera l'export vers Keras ou hls4ml **n'a pas PyTorch installé** : le
chargeur accepte indifféremment des tenseurs torch et des tableaux numpy, il ne
fait qu'un `np.asarray()` sur les valeurs.

In [5]:
out = pathlib.Path("hexnet_demo")
out.mkdir(exist_ok=True)

torch.save({"model_state_dict": net.state_dict(), "share_neighbors": "ring"},
           out / "hexnet.pth")

sd = net.state_dict()
np.savez(out / "hexnet.npz", **{k: v.detach().cpu().numpy() for k, v in sd.items()})

print("paramètres sauvegardés :")
for k, v in sd.items():
    print(f"  {k:22s} {tuple(v.shape)}")

paramètres sauvegardés :
  conv1.shared_weights   (4, 1, 2)
  conv1.bias_tensor      (4,)
  conv2.shared_weights   (8, 4, 2)
  conv2.bias_tensor      (8,)
  head.weight            (1, 8)
  head.bias              (1,)


## 3. Reconstruire côté Keras et charger les poids

Le modèle Keras est écrit à la main : c'est **à toi** de reproduire
l'architecture. Attention à la disposition — PyTorch travaille en
`(N, C, H, W)`, Keras en `(N, H, W, C)`.

Note aussi que `x.mean(dim=(2,3))` de PyTorch devient un
`GlobalAveragePooling2D`, et que le `Linear` final devient un `Dense`.

In [6]:
import keras
import keras_hexagdly as khex
from keras_hexagdly.torch_interop import load_torch_state_dict

def build_keras(share="ring"):
    inp = keras.Input(shape=(H, W, 1), name="image")
    x = khex.Conv2d(1, 4, kernel_size=1, stride=1, bias=True,
                    share_neighbors=share, name="conv1")(inp)
    x = keras.layers.ReLU()(x)
    x = khex.MaxPool2d(kernel_size=1, stride=2)(x)
    x = khex.Conv2d(4, 8, kernel_size=1, stride=1, bias=True,
                    share_neighbors=share, name="conv2")(x)
    x = keras.layers.ReLU()(x)
    x = keras.layers.GlobalAveragePooling2D()(x)
    return keras.Model(inp, keras.layers.Dense(1, name="head")(x))

kmodel = build_keras()
kmodel.summary()

2026-07-28 13:00:46.832022: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-28 13:00:46.876851: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


W0000 00:00:1785236448.517145  168307 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image (InputLayer)              │ (None, 16, 16, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1 (Conv2d)                  │ (None, 16, 16, 4)      │            12 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 16, 16, 4)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pool2d (MaxPool2d)          │ (None, 8, 8, 4)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv2d)                  │ (None, 8, 8, 8)        │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 8, 8, 8)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 8)              │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ head (Dense)                    │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 93 (372.00 B)

 Trainable params: 93 (372.00 B)

 Non-trainable params: 0 (0.00 B)

In [7]:
# Depuis le .npz : aucun import de torch n'est nécessaire ici.
weights = dict(np.load(out / "hexnet.npz"))
print("types dans le .npz :", sorted({type(v).__name__ for v in weights.values()}))

load_torch_state_dict(kmodel, weights, mapping={
    "conv1": "conv1",     # nom Keras -> préfixe PyTorch
    "conv2": "conv2",
    "head":  "head",
})
print("poids chargés")

types dans le .npz : ['ndarray']
poids chargés


Le `mapping` est explicite à dessein. Un appariement automatique par ordre
d'apparition échouerait **silencieusement** dès que les deux graphes divergent
un peu, et un poids mal transposé produit un modèle qui tourne, renvoie des
nombres plausibles, et donne de mauvais résultats.

Pour la même raison, toute couche portant des poids et absente du `mapping`
lève une exception plutôt que d'être ignorée — sinon elle garderait son
initialisation aléatoire sans que rien ne le signale.

## 4. Vérifier que les deux modèles concordent

L'étape à ne pas sauter.

In [8]:
net.eval()
with torch.no_grad():
    y_torch = net(x_test).numpy()

x_test_keras = np.transpose(x_test.numpy(), (0, 2, 3, 1))   # (N,C,H,W) -> (N,H,W,C)
y_keras = kmodel.predict(x_test_keras, verbose=0)

err = np.abs(y_torch - y_keras)
print(f"écart max   : {err.max():.3e}")
print(f"écart moyen : {err.mean():.3e}")
print(f"erreur relative max : {(err / (np.abs(y_torch) + 1e-9)).max():.3e}")

print(f"\n{'#':>3} {'PyTorch':>11} {'Keras':>11} {'écart':>10}")
for i in range(6):
    print(f"{i:>3} {y_torch[i,0]:>11.6f} {y_keras[i,0]:>11.6f} {err[i,0]:>10.2e}")

assert err.max() < 1e-4, "les deux modèles divergent"
print("\n=> concordance confirmée")

écart max   : 2.384e-06
écart moyen : 4.824e-07
erreur relative max : 8.768e-07

  #     PyTorch       Keras      écart
  0    2.316776    2.316776   0.00e+00
  1   -1.778669   -1.778669   0.00e+00
  2    1.164120    1.164120   0.00e+00
  3   -4.150315   -4.150314   9.54e-07
  4    1.978868    1.978869   1.19e-06
  5    1.951483    1.951483   7.15e-07

=> concordance confirmée


In [9]:
# Et les décisions, pas seulement les logits :
acc_torch = ((y_torch > 0) == y_test.numpy().astype(bool)).mean()
acc_keras = ((y_keras > 0) == y_test.numpy().astype(bool)).mean()
print(f"accuracy PyTorch : {acc_torch:.4f}")
print(f"accuracy Keras   : {acc_keras:.4f}")
assert acc_torch == acc_keras

accuracy PyTorch : 1.0000
accuracy Keras   : 1.0000


## Ce qu'il faut retenir

Un écart de l'ordre de `1e-6` sur des sorties d'amplitude ~1 est l'epsilon de la
simple précision : les deux modèles font le **même calcul**, seul l'ordre des
sommations diffère.

Ce que le chargeur prend en charge :

| | |
|---|---|
| `keras_hexagdly.Conv2d` | non partagé (`kernel0`, `kernel1`, …) **et** partagé (`shared_weights`) pour `ring` / `diag` / `sym` |
| `keras.layers.Dense` | transposition `(out, in) → (in, out)` |
| `keras.layers.Conv2D` | transposition `(out, in, kh, kw) → (kh, kw, in, out)` |

Il fonctionne aussi avec des checkpoints du **`hexagdly` amont**, qui nomme ses
paramètres exactement pareil dans le cas non partagé (il n'a pas de partage).

Ce qu'il ne fait **pas**, et ne peut pas faire : deviner la correspondance entre
les deux graphes, ni convertir une couche réécrite d'un type vers un autre — un
`Conv3d` reformulé en `Conv2D`, par exemple, reste à ta charge (le chargeur
lève une erreur explicite plutôt que de laisser passer).

La suite naturelle est l'export hls4ml : voir `keras_hexagdly_hls4ml_export.ipynb`.